### Note on data anonymization

According to the data sharing agreement with ECDC (Tessy data), case-based
information cannot be published or shared.


> Because some rows in the dataset contain only one case, this is considered
> case-based data and cannot be published or shared.

To comply with these legal restrictions, the original `Incidence` column is
replaced with **synthetic values** that maintain a similar overall distribution
(mean, variance, relative magnitudes) while ensuring:
- no individual value matches the original,
- case-based entries (e.g., incidence = 1) are removed,
- the column cannot be used to reconstruct Tessy data.

The resulting dataset preserves statistical properties for analysis and
visualization **without disclosing any protected ECDC/Tessy case-based data**.



All analyses shown in the repository rely on aggregated data products that respect the data
sharing agreement.  

For access to the original data or for more detailed information, **please
contact ECDC directly**.

In [2]:

import geopandas as gpd
import numpy as np

# --- PARAMETERS ---
noise_strength = 0.25   # relative noise level (25% of original value)

# --- LOAD SHAPEFILE ---
gdf = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")

# --- BACKUP ORIGINAL COLUMN ---
gdf["incidence_original"] = gdf["incidence"]

# --- GENERATE SYNTHETIC VALUES ---
# Add controlled noise + random permutation + enforce non-disclosure rules
orig = gdf["incidence"].values.astype(float)

# Step 1: add proportional noise (lognormal-like variability)
noise = np.random.normal(loc=0, scale=noise_strength, size=len(orig))
synthetic = orig * (1 + noise)

# Step 2: random small additive noise to avoid accidental identical values
synthetic += np.random.uniform(-0.5, 0.5, len(orig))

# Step 3: enforce non-disclosure: avoid zeros and ones (single-case)
synthetic = np.clip(synthetic, 2, None)   # lower bound = 2 cases

# Step 4: round to integers (if incidence is integer-based)
synthetic = synthetic.round().astype(int)

# Step 5: random permutation → breaks direct linkage
np.random.shuffle(synthetic)

# Save modified values
gdf["incidence"] = synthetic
gdf = gdf.drop(columns=["incidence_original"])





Q:\UserTemp\massaro\AppData\Local\Temp\130\ipykernel_149708\2600726930.py:28: RuntimeWarning: invalid value encountered in cast
  synthetic = synthetic.round().astype(int)


### Additional anonymization step: local neighbor-based swapping

To further ensure that the `Incidence` values cannot be traced back to the
original Tessy case-based data, we perform an additional anonymization step.
After generating the synthetic incidence values, we randomly **swap values
between each row and a small set of nearby rows** (e.g., within ±3 positions).

This approach introduces controlled randomness while maintaining a plausible
spatial structure. It ensures that:

- no row keeps its original incidence value,
- values remain in a similar spatial context (local consistency),
- the data cannot be reverse engineered to recover the protected Tessy data,
- the dataset remains suitable for visualization and aggregated analysis.


Combined with the synthetic re-generation of incidence values, this local
reshuffling reinforces privacy protection while keeping the data analytically
useful.


In [3]:
# number of neighbors to consider above or below
window = 3  

inc = gdf["incidence"].values.copy()

for i in range(len(inc)):
    # choose a nearby index
    low = max(0, i - window)
    high = min(len(inc) - 1, i + window)
    j = np.random.randint(low, high + 1)
    
    # swap i <-> j
    inc[i], inc[j] = inc[j], inc[i]

gdf["incidence"] = inc

In [4]:
# --- SAVE OUTPUT ---
gdf.to_file("../Data/Shapefiles/APES_metric.gpkg")